# Capstone — Análisis de interpretabilidad: ¿qué capturó la atención?

**Propósito:** exportar la evidencia de interpretabilidad para el informe final:
predicciones por documento del modelo Transformer + LoRA, cruce contra las predicciones
del baseline TF-IDF (mismo test set), y mapas de atención de los casos de desacuerdo.

Reentrena con la configuración EXACTA del Módulo 4 (misma semilla, mismos splits,
mismos hiperparámetros) y compara contra `reports/metrics/baseline_test_predictions.csv`
versionado en el repositorio.

> **Ejecutar en Google Colab con GPU** (Entorno de ejecución → Cambiar tipo → GPU T4).

In [ ]:
!pip install -q "transformers>=4.41" "datasets>=2.19" "peft>=0.10" "accelerate>=0.30"
# torchao 0.10 preinstalado en Colab es incompatible con peft reciente; no lo usamos
!pip uninstall -q -y torchao

In [ ]:
import torch, time, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("GPU disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
assert torch.cuda.is_available(), "Activar GPU: Entorno de ejecución -> Cambiar tipo de entorno"

In [ ]:
import os
import pandas as pd

if not os.path.exists("nlp-capstone-agnews"):
    !git clone -q https://github.com/frncsco33/nlp-capstone-agnews.git
REPO = "nlp-capstone-agnews"

LABEL_NAMES = ["World", "Sports", "Business", "Sci_Tech"]
LABEL2ID = {name: i for i, name in enumerate(LABEL_NAMES)}

train_full = pd.read_csv(f"{REPO}/data/ag_news/ag_news_train.csv")
test = pd.read_csv(f"{REPO}/data/ag_news/ag_news_test.csv")
for df in (train_full, test):
    df["labels"] = df["label"].map(LABEL2ID)

from sklearn.model_selection import train_test_split
tr, val = train_test_split(train_full, test_size=0.1, random_state=SEED,
                           stratify=train_full["labels"])
print(f"train: {len(tr):,} | val: {len(val):,} | test: {len(test):,}")

base_pred = pd.read_csv(f"{REPO}/reports/metrics/baseline_test_predictions.csv")
assert (base_pred["label"].values == test["labels"].values).all(), "desalineado vs baseline"

## 1. Reentrenamiento idéntico al Módulo 4

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

subword_lens = [len(ids) for ids in tokenizer(train_full["text"].tolist(),
                                              truncation=False)["input_ids"]]
MAX_LEN = int(np.percentile(subword_lens, 95))
print("max_len (p95 subword):", MAX_LEN)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

cols = ["text", "labels"]
ds_train = Dataset.from_pandas(tr[cols], preserve_index=False).map(tokenize, batched=True)
ds_val = Dataset.from_pandas(val[cols], preserve_index=False).map(tokenize, batched=True)
ds_test = Dataset.from_pandas(test[cols], preserve_index=False).map(tokenize, batched=True)

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABEL_NAMES),
    id2label={i: n for i, n in enumerate(LABEL_NAMES)}, label2id=LABEL2ID,
    attn_implementation="eager")  # sdpa no expone output_attentions; eager sí
model = get_peft_model(base_model, LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"], bias="none"))
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir="outputs/lora", learning_rate=2e-4,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    num_train_epochs=3, eval_strategy="epoch", save_strategy="no",
    logging_strategy="epoch", fp16=True, seed=SEED, report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer))
t0 = time.time()
trainer.train()
print(f"tiempo: {time.time()-t0:.1f}s")

## 2. Predicciones sobre el test y cruce con el baseline

In [ ]:
from sklearn.metrics import f1_score

logits = trainer.predict(ds_test).predictions
pred_lora = logits.argmax(-1)
conf_lora = (np.exp(logits) / np.exp(logits).sum(-1, keepdims=True)).max(-1)
f1 = f1_score(test["labels"], pred_lora, average="weighted")
print(f"F1 weighted test de esta corrida: {f1:.4f} (referencia M4: 0.9101)")

df = base_pred.copy()
df["pred_lora"] = pred_lora
df["conf_lora"] = conf_lora.round(4)
df.to_csv("lora_test_predictions.csv", index=False)

ok_t = df["pred_tfidf"] == df["label"]
ok_l = df["pred_lora"] == df["label"]
mcnemar = {"ambos_aciertan": int((ok_t & ok_l).sum()),
           "solo_tfidf": int((ok_t & ~ok_l).sum()),
           "solo_lora": int((~ok_t & ok_l).sum()),
           "ambos_fallan": int((~ok_t & ~ok_l).sum())}
print("tabla de desacuerdo (McNemar):", mcnemar)

## 3. Mapas de atención de los casos de desacuerdo

Se seleccionan ejemplos donde **TF-IDF falla y LoRA acierta** (priorizando el par
Business↔Sci_Tech, la frontera difícil) y algunos donde **ambos fallan**. Para cada uno
se guarda la atención del token [CLS] hacia cada token (última capa, promedio de las
12 cabezas): es la vista de "qué miró" la representación que decide la clase.

In [ ]:
def cls_attention(text):
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(model.device)
    model.eval()
    with torch.no_grad():
        out = model(**enc, output_attentions=True)
    assert out.attentions, "el modelo debe cargarse con attn_implementation='eager'"
    att = out.attentions[-1][0].mean(0)[0]          # última capa, media de cabezas, fila CLS
    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    pred = int(out.logits.argmax(-1))
    return toks, att.float().cpu().numpy().round(4).tolist(), pred

solo_lora = df[~ok_t & ok_l].copy()
solo_lora["par_dificil"] = (
    solo_lora["label"].isin([2, 3]) & solo_lora["pred_tfidf"].isin([2, 3]))
candidatos = pd.concat([
    solo_lora[solo_lora["par_dificil"]].nlargest(8, "conf_lora"),
    solo_lora[~solo_lora["par_dificil"]].nlargest(6, "conf_lora"),
])
ambos_mal = df[~ok_t & ~ok_l].nlargest(4, "conf_lora")

ejemplos = []
for tipo, subset in [("tfidf_falla_lora_acierta", candidatos), ("ambos_fallan", ambos_mal)]:
    for _, row in subset.iterrows():
        i = int(row["row_id"])
        toks, att, _ = cls_attention(test["text"].iloc[i])
        ejemplos.append({
            "tipo": tipo, "row_id": i,
            "texto": test["text"].iloc[i][:400],
            "label": LABEL_NAMES[int(row["label"])],
            "pred_tfidf": LABEL_NAMES[int(row["pred_tfidf"])],
            "pred_lora": LABEL_NAMES[int(row["pred_lora"])],
            "conf_tfidf": float(row["conf_tfidf"]), "conf_lora": float(row["conf_lora"]),
            "tokens": toks, "atencion_cls": att,
        })
print(f"ejemplos exportados: {len(ejemplos)}")

In [ ]:
resumen = {"f1_weighted_test": round(float(f1), 4), "mcnemar": mcnemar,
           "max_len": MAX_LEN, "gpu": torch.cuda.get_device_name(0)}
with open("capstone_run.json", "w") as f:
    json.dump(resumen, f, indent=2)
with open("attention_examples.json", "w") as f:
    json.dump(ejemplos, f, indent=2, ensure_ascii=False)

!zip -q resultados_capstone.zip lora_test_predictions.csv attention_examples.json capstone_run.json
from google.colab import files; files.download("resultados_capstone.zip")
print("Listo: descargar resultados_capstone.zip")